# 01b · Schema Decisions

Follows `01_data_profiling.ipynb`. Reads `Outputs/schema_matrix.csv`, so it runs standalone.

Answers four questions that gate the ingestion pipeline:

1. Which columns appear at 2024 (expected: the BESS install)?
2. What is the timestamp format, coverage and true sampling interval? Are there duplicates?
3. Which of the 14 stable columns are cumulative counters and which are instantaneous?
4. What is in the Master Meter 1 reference file, and can it support benchmark parity?

Writes `Outputs/schema_drift_summary.md`, cited in the Assessment 2 report.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

REPO = Path("/Users/uttamshrestha/Desktop/Data Science Practice Unit/PRT661-solar-forecasting")
RAW, REF, OUT = REPO/"Datasets"/"raw", REPO/"Datasets"/"reference", REPO/"Outputs"

matrix = pd.read_csv(OUT/"schema_matrix.csv", index_col=0)
stable = matrix.index[matrix.all(axis=1)].tolist()
print(f"{matrix.shape[0]} columns x {matrix.shape[1]} years | {len(stable)} stable")

406 columns x 19 years | 14 stable


## 1 · What appears at 2024?

In [2]:
y23, y24 = "Alice_Springs_2023", "Alice_Springs_2024"
added = matrix.index[~matrix[y23] & matrix[y24]].tolist()
lost  = matrix.index[ matrix[y23] & ~matrix[y24]].tolist()

print(f"NEW IN 2024 ({len(added)}):")
for c in added:
    print("   ", c)
print(f"\nGONE AFTER 2023 ({len(lost)}):")
for c in lost:
    print("   ", c)

NEW IN 2024 (13):
    239_DKA_Totals_BESS_Reactive_Power
    239_DKA_Totals_BESS_Active_Power
    239_DKA_Totals_BESS_Apparent_Power
    239_DKA_Totals_BESS_State_of_Charge
    242_DKA_Totals_Grid_Reactive_Power
    242_DKA_Totals_Grid_Active_Power
    242_DKA_Totals_Grid_Apparent_Power
    241_DKA_Totals_PV_Reactive_Power
    241_DKA_Totals_PV_Active_Power
    241_DKA_Totals_PV_Apparent_Power
    240_DKA_Totals_Site_Demand_Reactive_Power
    240_DKA_Totals_Site_Demand_Active_Power
    240_DKA_Totals_Site_Demand_Apparent_Power

GONE AFTER 2023 (0):


## 2 · Timestamps: format, coverage, interval, duplicates

Reads only the timestamp column, so this is fast even on the 223 MB files.

In [3]:
for name in ["Alice_Springs_2008", "Alice_Springs_2015", "Alice_Springs_2025", "Alice_Springs_2026"]:
    ts = pd.read_csv(RAW/f"{name}.csv", usecols=["timestamp"])["timestamp"]
    print(f"--- {name} ---")
    print("   raw first value :", repr(ts.iloc[0]))
    t = pd.to_datetime(ts, errors="coerce")
    print("   unparsed        :", int(t.isna().sum()))
    print("   range           :", t.min(), "to", t.max())
    d = t.diff().dropna().value_counts().head(4)
    print("   interval counts :")
    for k, v in d.items():
        print(f"       {k}  x{v:,}")
    dup = int(t.duplicated().sum())
    print("   duplicate stamps:", f"{dup:,}")
    if dup:
        print("   example dups    :", list(t[t.duplicated(keep=False)].head(4)))
    print()

--- Alice_Springs_2008 ---
   raw first value : '2008-09-12 05:55:00'
   unparsed        : 0
   range           : 2008-09-12 05:55:00 to 2009-01-01 23:55:00
   interval counts :
       0 days 00:05:00  x32,042
       0 days 07:05:00  x1
       0 days 00:25:00  x1
       0 days 04:20:00  x1
   duplicate stamps: 0

--- Alice_Springs_2015 ---
   raw first value : '2015-01-01 00:00:00'
   unparsed        : 0
   range           : 2015-01-01 00:00:00 to 2016-01-01 23:55:00
   interval counts :
       0 days 00:05:00  x105,407
   duplicate stamps: 0

--- Alice_Springs_2025 ---
   raw first value : '2025-01-01 00:00:00'
   unparsed        : 0
   range           : 2025-01-01 00:00:00 to 2026-01-01 23:55:00
   interval counts :
       0 days 00:05:00  x105,360
       0 days 00:10:00  x17
       0 days 00:04:59  x2
       0 days 00:00:03  x2
   duplicate stamps: 0

--- Alice_Springs_2026 ---
   raw first value : '2026-01-01 00:00:00'
   unparsed        : 0
   range           : 2026-01-01 00:00:00

## 3 · Are the stable columns cumulative or instantaneous?

The distinction decides whether the pipeline differences a counter or uses the value directly. A monotonic non decreasing series is a counter.

In [4]:
probe = "Alice_Springs_2025"
df = pd.read_csv(RAW/f"{probe}.csv", usecols=stable, parse_dates=["timestamp"]).sort_values("timestamp")
print(f"{probe}: {len(df):,} rows\n")

rows = []
for c in [c for c in stable if c != "timestamp"]:
    s = pd.to_numeric(df[c], errors="coerce")
    diffs = s.diff().dropna()
    rows.append({
        "column":     c,
        "missing_%":  round(100 * s.isna().mean(), 2),
        "min":        round(s.min(), 3) if s.notna().any() else None,
        "max":        round(s.max(), 3) if s.notna().any() else None,
        "mean":       round(s.mean(), 3) if s.notna().any() else None,
        "neg_diff_%": round(100 * (diffs < 0).mean(), 2),
        "looks_like": "CUMULATIVE" if (diffs < 0).mean() < 0.01 else "instantaneous",
    })
summary = pd.DataFrame(rows)
pd.set_option("display.width", 200, "display.max_colwidth", 60)
print(summary.to_string(index=False))

Alice_Springs_2025: 105,393 rows

                                                                 column  missing_%       min       max      mean  neg_diff_%    looks_like
205_Archived_DKA_M15_BPhase_UMG_QCells_Active_Energy_Delivered_Received      100.0       NaN       NaN       NaN         NaN instantaneous
           205_Archived_DKA_M15_BPhase_UMG_QCells_Current_Phase_Average      100.0       NaN       NaN       NaN         NaN instantaneous
                    205_Archived_DKA_M15_BPhase_UMG_QCells_Active_Power      100.0       NaN       NaN       NaN         NaN instantaneous
                    100_DKA_M1_A_Phase_Active_Energy_Delivered_Received        0.2 53977.000 56955.000 55459.602        0.00    CUMULATIVE
                               100_DKA_M1_A_Phase_Current_Phase_Average        0.2     0.000     6.830     1.490       22.24 instantaneous
                                        100_DKA_M1_A_Phase_Active_Power        0.2    -0.000     1.641     0.337       22.77 instant

/var/folders/wv/gp0tk8_13p56d5w4twffnxz40000gn/T/ipykernel_13423/3647277934.py:2: DtypeWarning: Columns (23) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(RAW/f"{probe}.csv", usecols=stable, parse_dates=["timestamp"]).sort_values("timestamp")


## 4 · Master Meter 1 reference file

The published benchmark reports on Master Meter 1 and Master Meter 2. Establishing what this file contains determines whether a like for like comparison is possible.

In [5]:
for f in sorted(REF.glob("*.csv")):
    head = pd.read_csv(f, nrows=5)
    print(f"=== {f.name}  ({f.stat().st_size/1e6:.0f} MB, {head.shape[1]} columns) ===")
    for c in head.columns:
        print("   ", c)
    print()
    print(head.to_string())
    print()
    ts_col = [c for c in head.columns if "time" in c.lower() or "date" in c.lower()]
    if ts_col:
        ts = pd.to_datetime(pd.read_csv(f, usecols=ts_col[:1])[ts_col[0]], errors="coerce")
        print(f"   coverage: {ts.min()} to {ts.max()}  ({len(ts):,} rows)")
        print(f"   modal interval: {ts.diff().mode().iloc[0]}")

=== 96-Site_DKA-MasterMeter1.csv  (390 MB, 17 columns) ===
    timestamp
    Active_Energy_Delivered_Received
    Current_Phase_Average
    Active_Power
    Power_Factor_Signed
    Average_Voltage_Line_to_Neutral
    Frequency
    THD_Voltage_Average
    Wind_Speed
    Weather_Temperature_Celsius
    Weather_Relative_Humidity
    Global_Horizontal_Radiation
    Diffuse_Horizontal_Radiation
    Wind_Direction
    Weather_Daily_Rainfall
    Radiation_Global_Tilted
    Radiation_Diffuse_Tilted

             timestamp  Active_Energy_Delivered_Received  Current_Phase_Average  Active_Power  Power_Factor_Signed  Average_Voltage_Line_to_Neutral  Frequency  THD_Voltage_Average  Wind_Speed  Weather_Temperature_Celsius  Weather_Relative_Humidity  Global_Horizontal_Radiation  Diffuse_Horizontal_Radiation  Wind_Direction  Weather_Daily_Rainfall  Radiation_Global_Tilted  Radiation_Diffuse_Tilted
0  2008-09-12 05:55:00                         -2.638068                    NaN     -0.245161            

## 5 · Write the drift summary for the report

In [6]:
lines = [
    "# Schema Drift Summary",
    "",
    "Generated by `notebooks/01b_schema_decisions.ipynb`. Source: 19 raw DKASC files, 2008 to 2026.",
    "",
    f"- Total rows across all years: **{1824033:,}**",
    f"- Union of column names: **{matrix.shape[0]}**",
    f"- Columns present in every year: **{len(stable)}**",
    f"- Columns exhibiting drift: **{matrix.shape[0] - len(stable)}**",
    f"- Columns appearing at 2024: **{len(added)}**",
    "",
    "## Columns present in every year (2008 to 2026)",
    "",
]
lines += [f"- `{c}`" for c in stable]
lines += ["", "## Columns appearing at 2024", ""]
lines += [f"- `{c}`" for c in added]
lines += ["", "## Coverage of every column", "", "| Column | Years present | First | Last |", "|---|---|---|---|"]
for c in matrix.index:
    yrs = [y.replace("Alice_Springs_", "") for y in matrix.columns if matrix.loc[c, y]]
    lines.append(f"| `{c}` | {len(yrs)} | {yrs[0]} | {yrs[-1]} |")

(OUT/"schema_drift_summary.md").write_text("\n".join(lines))
print(f"wrote {OUT/'schema_drift_summary.md'} ({len(lines)} lines)")

wrote /Users/uttamshrestha/Desktop/Data Science Practice Unit/PRT661-solar-forecasting/Outputs/schema_drift_summary.md (453 lines)
